# 《转生千金》Colab Pro+ 后台执行 · 零弹窗云端全自动美术工厂

- **零弹窗存储**：全程不调用 `drive.mount()`！通过左侧边栏 **🔑 Secrets** 里的 `HF_TOKEN` + `HF_REPO_ID` 高速直连你的 **Hugging Face 私有仓库**，多开 1～3 个标签页并行也不会冲突。
- **关电脑继续跑**：在顶部菜单开启 **`代码执行程序 → 后台执行`** 后点击 **`全部运行`**，即可合盖或关机，跑完自动调用 `runtime.unassign()` 释放实例停表，绝不烧空算力。
- **所需 Secrets（左侧 🔑 配一次，所有标签页永久复用）**：
  1. `HF_TOKEN`：Hugging Face Write Token
  2. `HF_REPO_ID`：私有 Dataset 仓库名（如 `yourname/fictional-adventure-art-cloud`）
  3. `CODEX_AUTH_B64`：`chatgpt-2` 的 `auth.json` Base64（运行本地 `tools/cloud_vps/sync_to_hf_colab.ps1` 自动复制到剪贴板，粘贴一次即可）


In [ ]:
# ==================== Cell 1: 零弹窗初始化 + chatgpt-2 分片生图 ====================
# @title 1. 启动零弹窗云端生图（多开并行可改 SHARD，如 "1/3"、"2/3"、"3/3"）
SHARD = "1/1"          # @param {type:"string"}
PHASE = ""             # @param ["", "vol3_bg", "vol3_cg", "vol2_3_portraits", "vol2_3_diffs"]
LIMIT = 0              # @param {type:"integer"}  # 0 = 跑完当前分片全部 pending 项
RUN_GPU_DENOISE = True # @param {type:"boolean"}  # 若挂载了 GPU，顺带自动跑 DRUNet 去噪精修
AUTO_UNASSIGN = True   # @param {type:"boolean"}  # 跑完自动销毁实例停止计费（后台执行强烈建议开启）

import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import userdata

def _need_secret(key: str) -> str:
    val = (userdata.get(key) or "").strip()
    if not val:
        raise RuntimeError(f"请在 Colab 左侧边栏 🔑 Secrets 中配置并勾选允许访问: {key}")
    return val

HF_TOKEN = _need_secret("HF_TOKEN")
HF_REPO_ID = _need_secret("HF_REPO_ID")
CODEX_AUTH_B64 = _need_secret("CODEX_AUTH_B64")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_REPO_ID"] = HF_REPO_ID
os.environ["HF_REPO_TYPE"] = "dataset"
os.environ["FA_DRIVE_BACKEND"] = "hf"

# 1. 仅按需补齐轻量依赖（严禁升级 Colab 预装的 numpy / torch / pillow 破坏 C-ABI！）
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", "huggingface_hub"], check=True)

# 2. 安装 Node.js 22 + 官方 @openai/codex CLI
if not shutil.which("codex"):
    print("📦 正在安装 Node.js 22 与 @openai/codex CLI...")
    subprocess.run("curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y -qq nodejs", shell=True, check=True)
    subprocess.run(["npm", "install", "-g", "@openai/codex"], check=True)
print("✅ codex:", subprocess.check_output(["codex", "--version"], text=True).strip())

# 3. 还原加密的 chatgpt-2 auth.json（隔离目录 + 最小零 MCP 配置）
CODEX_HOME = Path("/root/.codex-chatgpt2")
CODEX_HOME.mkdir(parents=True, exist_ok=True)
auth_bytes = base64.b64decode(CODEX_AUTH_B64)
json.loads(auth_bytes.decode("utf-8"))  # 校验 JSON 完整性
(CODEX_HOME / "auth.json").write_bytes(auth_bytes)
os.chmod(CODEX_HOME / "auth.json", 0o600)
(CODEX_HOME / "config.toml").write_text("# Minimal cloud config (zero desktop MCPs)\n", encoding="utf-8")
os.environ["CODEX_IMGGEN_HOME"] = str(CODEX_HOME)
os.environ["CODEX_HOME"] = str(CODEX_HOME)

# 4. 从你的私有 Hugging Face 仓库拉取最新仓库快照与台账（0 Drive 弹窗！）
from huggingface_hub import HfApi, hf_hub_download
api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_REPO_ID, repo_type="dataset", private=True, exist_ok=True)

WORK_REPO = Path("/content/fictional-adventure")
if WORK_REPO.exists():
    shutil.rmtree(WORK_REPO)

repo_files = set(api.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset"))
bundle_remote = next((f for f in ("repo/fictional-adventure.bundle", "fictional-adventure.bundle") if f in repo_files), None)
if bundle_remote:
    print(f"📥 从 HF 私有库拉取最新 {bundle_remote}（含未推送的本地提交）...")
    bundle_path = hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename=bundle_remote, token=HF_TOKEN)
    subprocess.run(["git", "clone", bundle_path, str(WORK_REPO)], check=True)
else:
    print("📥 从 GitHub 克隆 fictional-adventure...")
    subprocess.run(["git", "clone", "https://github.com/juanhanyu/fictional-adventure.git", str(WORK_REPO)], check=True)

mf_remote = next((f for f in ("manifests/production_manifest.json", "production_manifest.json") if f in repo_files), None)
if mf_remote:
    mf_dl = hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename=mf_remote, token=HF_TOKEN)
    dst_mf = WORK_REPO / "tools/art_pipeline/production_manifest.json"
    dst_mf.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(mf_dl, dst_mf)
    print(f"📋 已同步最新 {mf_remote}")

for fname in ("batch_runner.py", "codex_imggen.py"):
    cand = next((f for f in (f"tools/art_pipeline/{fname}", fname) if f in repo_files), None)
    if cand:
        dl_f = hf_hub_download(repo_id=HF_REPO_ID, repo_type="dataset", filename=cand, token=HF_TOKEN)
        dst_f = WORK_REPO / "tools/art_pipeline" / fname
        dst_f.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(dl_f, dst_f)
        print(f"🧩 已覆写最新脚本 {cand} -> {dst_f.name}")

# 5. 启动断点续传生图（每完成 1 张立刻上传 HF 私有库 + 写独立回执）
cmd = [sys.executable, str(WORK_REPO / "tools/art_pipeline/batch_runner.py"), "--resume"]
if SHARD and SHARD != "1/1":
    cmd.extend(["--shard", SHARD])
if PHASE:
    cmd.extend(["--phase", PHASE])
if LIMIT > 0:
    cmd.extend(["--limit", str(LIMIT)])

print("🚀 开始运行:", " ".join(cmd))
subprocess.run(cmd, cwd=str(WORK_REPO), check=False)
print("✅ Cell 1 生图阶段完成！")


In [ ]:
# ==================== Cell 2: 可选 GPU 去噪精修 + 自动销毁停表 ====================
# @title 2. 自动对本批新图执行 KeepColor GPU 去噪并释放实例停表
import os, shutil, subprocess, sys
from pathlib import Path

WORK_REPO = Path("/content/fictional-adventure")
RAW_DIR = WORK_REPO / "tools/art_pipeline/raw"
DENOISED_DIR = Path("/content/denoised_ready")
DENOISED_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    has_gpu = torch.cuda.is_available()
except Exception:
    has_gpu = False

if RUN_GPU_DENOISE and has_gpu and RAW_DIR.exists():
    new_pngs = sorted(RAW_DIR.glob("*.png"))
    if new_pngs:
        print(f"⚡ 检测到 GPU ({torch.cuda.get_device_name(0)})，开始对本轮 {len(new_pngs)} 张原图执行 keepcolor-denoise...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", "spandrel"], check=True)
        kc_dir = Path("/content/keepcolor-denoise")
        if not kc_dir.exists():
            subprocess.run(["git", "clone", "https://github.com/juanhanyu/keepcolor-denoise.git", str(kc_dir)], check=True)
        from huggingface_hub import HfApi
        api = HfApi(token=os.environ["HF_TOKEN"])
        for png in new_pngs:
            out_png = DENOISED_DIR / png.name
            sigma = "8.0" if png.name.startswith("portrait_") else "9.0"
            res = subprocess.run([sys.executable, str(kc_dir / "denoise.py"), str(png), str(out_png), "--sigma", sigma], check=False)
            if res.returncode == 0 and out_png.exists():
                api.upload_file(
                    path_or_fileobj=str(out_png),
                    path_in_repo=f"denoised_ready/{out_png.name}",
                    repo_id=os.environ["HF_REPO_ID"],
                    repo_type="dataset",
                    commit_message=f"keepcolor-denoise: {out_png.name}",
                )
                print(f"  ✨ 已上传去噪成品: denoised_ready/{out_png.name}")
    else:
        print("ℹ️ 本轮无新增本地原图需要去噪。")
else:
    print(f"ℹ️ 跳过 GPU 去噪（RUN_GPU_DENOISE={RUN_GPU_DENOISE}, has_gpu={has_gpu}）。")

if AUTO_UNASSIGN:
    print("🛑 全部任务已安全同步至 Hugging Face 私有库！5 秒后自动销毁 Colab 实例停止计费...")
    import time
    time.sleep(5)
    from google.colab import runtime
    runtime.unassign()
